In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from dataforge.src import DataDict
from dataforge.scripts.fps import furthest_point_sampling
from dataforge.src.generic import read_h5_file
from multiprocessing import Pool
import glob

folder_path = '/storage_common/angiod/MB-Fit-Data-Forge/POPC/POPC.MULTITEMP/data/xyz_capped/dimers/C-CCHH.C-CHHH'
N_SAMPLES = 1000

def update_lower_degree_nmer_sampled_idcs(filename):
    # Define the root folders
    root_folders = list(DataDict.FOLDER_NAMES.values())

    def get_root_folder(filename):
        for part in reversed(filename.split('/')):
            if part in root_folders:
                return part
        raise ValueError("Root folder not found in filename")

    # Extract the root folder from the filename
    root_folder = get_root_folder(filename)
    sampled_indices = np.loadtxt(filename, dtype=int)
    root_folder2num_monomers = {v: k for k, v in DataDict.FOLDER_NAMES.items()}
    nmers_idcs = os.path.basename(filename).split('.')[-2].split('_')[-root_folder2num_monomers[root_folder]:]

    # Get the base directory
    base_dir = filename.split(root_folder)[0] + root_folder

    # Find subfolders and files
    for subfolder in os.listdir(base_dir):
        subfolder_path = os.path.join(base_dir, subfolder)
        if os.path.isdir(subfolder_path) and subfolder in root_folders:
            for file in glob.glob(os.path.join(subfolder_path, '**/*.h5'), recursive=True):
                    file_nmers_idcs = os.path.basename(file).split('.')[-2].split('_')[-root_folder2num_monomers[get_root_folder(file)]:]
                    if any(idx in file_nmers_idcs for idx in nmers_idcs):
                        list_file = file.replace('.h5', '.list')
                        list_file_path = os.path.join(subfolder_path, list_file)
                        if os.path.exists(list_file_path):
                            existing_indices = np.loadtxt(list_file_path, dtype=int)
                            updated_indices = np.unique(np.concatenate((existing_indices, sampled_indices)))
                            np.savetxt(list_file_path, updated_indices, fmt='%d')
                        else:
                            np.savetxt(list_file_path, sampled_indices, fmt='%d')

def process_h5_file(h5_filename):
    coords, atom_types, fullnames, info_dict, extra_data = read_h5_file(h5_filename)
    names = extra_data['symmetry_names_sorted']
    sampled_indices = furthest_point_sampling(N_SAMPLES, coords, names, chunk_max_dim=1000000, max_symm_perm=100)
    
    def get_list_filename(h5_filename):
        base, _ = os.path.splitext(h5_filename)
        return base + '.list'
    
    output_filepath = get_list_filename(h5_filename)
    np.savetxt(output_filepath, sampled_indices, fmt='%d')
    update_lower_degree_nmer_sampled_idcs(output_filepath)

h5_files = glob.glob(os.path.join(folder_path, '*.h5'))

with Pool(processes=min(64, len(h5_files))) as pool:
    pool.map(process_h5_file, h5_files)